# Lab 1 — Databricks Basics and DEV Workflow

## Import data into Databricks (CSV)

In [0]:
catalog = "dbr_dev"
schema = "ivanrazumovskyi"

In [0]:
netflix_df = spark.read.csv(
    "/Volumes/dbr_dev/ivanrazumovskyi/datasets/netflix_titles.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

display(netflix_df)
print("Row count:", netflix_df.count())

## Practice basic Spark DataFrame operations (select, filter, join, groupBy)

In [0]:
# SELECT — choose relevant columns
selected_df = netflix_df.select("show_id", "type", "title", "release_year", "country", "rating")
display(selected_df)

In [0]:
# FILTER — keep only Movies released in 2015 or later
filtered_df = selected_df.filter((netflix_df.type == "Movie") & (netflix_df.release_year >= 2015))
display(filtered_df)

In [0]:
# GROUP BY — count titles per rating category
rating_summary_df = (
    netflix_df
    .groupBy("rating")
    .count()
    .orderBy("count", ascending=False)
)
display(rating_summary_df)

In [0]:
# Small reference DataFrame for join: rating -> audience category
rating_categories = spark.createDataFrame([
    ("G", "General Audience"),
    ("PG", "Parental Guidance"),
    ("PG-13", "Parental Guidance 13+"),
    ("R", "Restricted"),
    ("TV-Y", "Kids"),
    ("TV-Y7", "Kids 7+"),
    ("TV-Y7-FV", "Kids 7+ (Fantasy Violence)"),
    ("TV-G", "General Audience"),
    ("TV-PG", "Parental Guidance"),
    ("TV-14", "Teens 14+"),
    ("TV-MA", "Mature Audience"),
    ("NC-17", "Adults Only"),
    ("NR", "Not Rated"),
    ("UR", "Unrated"),
], ["rating", "audience_category"])

# JOIN — enrich netflix data with audience category
joined_df = netflix_df.join(rating_categories, on="rating", how="left")
display(joined_df.select("show_id", "title", "type", "rating", "audience_category"))

## Load data from an external API

In [0]:
import requests

# Fetch a page of TV shows from TVMaze API
response = requests.get("https://api.tvmaze.com/shows", params={"page": 0})
response.raise_for_status()

tvmaze_data = response.json()

tvmaze_shows = [
    {
        "tvmaze_id": show["id"],
        "name": show.get("name"),
        "status": show.get("status"),
        "rating": float(show["rating"]["average"]) if show.get("rating", {}).get("average") is not None else None,
        "genres": ", ".join(show.get("genres", [])),
        "language": show.get("language"),
        "premiered": show.get("premiered")
    }
    for show in tvmaze_data
]

tvmaze_df = spark.createDataFrame(tvmaze_shows)
top_rated_df = tvmaze_df.filter(tvmaze_df.rating.isNotNull()).orderBy(tvmaze_df.rating.desc())

display(top_rated_df)

## Load the data into a Delta table

In [0]:

full_results_table = f"{catalog}.{schema}.results"
# Save the main processed result as a Delta table
joined_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_results_table)

print(f"Saved to {full_results_table}")

In [0]:
# Verify the Delta table is queryable
result_check = spark.sql(f"SELECT * FROM {full_results_table} LIMIT 10")
display(result_check)

print("Total rows in results table:", spark.table(full_results_table).count())

## Build a basic dashboard over your table

Dashboard published in Databricks SQL: **"ivanrazumovskyi - Lab1 Netflix Dashboard"**

🔗 [Open Dashboard](https://adb-7405604503619901.1.azuredatabricks.net/dashboardsv3/01f17ace88f410e28ae8f743f89c0dc3/published?o=7405604503619901)

## Delta Lake Benefits

Delta Lake, used under the hood for our `results` table, provides several important benefits:

- **ACID transactions** — reliable and consistent reads and writes, even with concurrent access.
- **Schema enforcement & evolution** — prevents invalid data from being written, 
  while still allowing safe schema changes (e.g. via `overwriteSchema`).
- **Time travel** — every write creates a new table version, queryable via 
  `DESCRIBE HISTORY` or `VERSION AS OF`, allowing rollback and auditing.
- **Reliable batch processing** — notebooks can be re-run on the shared cluster 
  and safely overwrite the results table without corrupting data.